In [1]:
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

c:\Users\xyliz\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


2.11.0+cu128 True NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
pip install numpy

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 4.8 MB/s eta 0:00:03
   -- ------------------------------------- 0.8/12.6 MB 4.8 MB/s eta 0:00:03
   -- ------------------------------------- 0.8/12.6 MB 4.8 MB/s eta 0:00:03
   --- ------------------------------------ 1.0/12.6 MB 1.2 MB/s eta 0:00:10
   ----- ---------------------------------- 1.6/12.6 MB 1.6 MB/s eta 0:00:07
   ----- ---------------------------------- 1.6/12.6 MB 1.6 MB/s eta 0:00:07
   -------- ------------------------------- 2.6/12.6 MB 1.8 MB/s eta 0:00:06
   --------- ------------------------------ 2.9/12.6 MB 1.7 MB/s eta 0:00:06
   ----------- ---------------------------- 3.7/12.6 MB 1.9 MB/s eta 0:00:05
   ------------- -------------------------- 4.2/12.6 MB 2.0 MB/s eta 0:00:05
   -------------- ------------------------- 4.5/12.6 MB 2.0 MB/s eta 0:00:05
   --------------- ------------------------ 5.0/12.6 MB 2.0 MB/s eta 0:00:04
   ---


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import random

def load_pairs(path):
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            num, text = line.split("\t")
            pairs.append((num, text))
    return pairs

num_en = load_pairs("num2txt.txt")
num_ko = load_pairs("num2kor.txt")

print(len(num_en), len(num_ko))
for p in random.sample(num_en, 3):
    print(p)
for p in random.sample(num_ko, 3):
    print(p)

def tok_num(s):
    return list(s)

def tok_en(s):
    return s.split()

def tok_ko(s):
    return list(s)

print(tok_num("12345"))
print(tok_en("twelve thousand three hundred forty five"))
print(tok_ko("만 이천삼백사십오"))

en_vocab = sorted({w for _, t in num_en for w in tok_en(t)})
ko_vocab = sorted({c for _, t in num_ko for c in tok_ko(t)})
print(len(en_vocab), en_vocab)
print(len(ko_vocab), ko_vocab)
print(max(len(tok_en(t)) for _, t in num_en), max(len(tok_ko(t)) for _, t in num_ko))

100000 100000
('29043691', 'twenty nine million forty three thousand six hundred ninety one')
('1781342899', 'one billion seven hundred eighty one million three hundred forty two thousand eight hundred ninety nine')
('161176', 'one hundred sixty one thousand one hundred seventy six')
('8359681', '팔백삼십오만 구천육백팔십일')
('1051950507', '십억 오천백구십오만 오백칠')
('1130905', '백십삼만 구백오')
['1', '2', '3', '4', '5']
['twelve', 'thousand', 'three', 'hundred', 'forty', 'five']
['만', ' ', '이', '천', '삼', '백', '사', '십', '오']
32 ['billion', 'eight', 'eighteen', 'eighty', 'eleven', 'fifteen', 'fifty', 'five', 'forty', 'four', 'fourteen', 'hundred', 'million', 'nine', 'nineteen', 'ninety', 'one', 'seven', 'seventeen', 'seventy', 'six', 'sixteen', 'sixty', 'ten', 'thirteen', 'thirty', 'thousand', 'three', 'twelve', 'twenty', 'two', 'zero']
16 [' ', '구', '만', '백', '사', '삼', '십', '억', '영', '오', '육', '이', '일', '천', '칠', '팔']
16 21


In [4]:
PAD, SOS, EOS, UNK = 0, 1, 2, 3

class Vocab:
    def __init__(self, tokens, joiner):
        self.itos = ["<pad>", "<sos>", "<eos>", "<unk>"] + sorted(set(tokens))
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.joiner = joiner

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [SOS] + [self.stoi.get(t, UNK) for t in tokens] + [EOS]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return self.joiner.join(out)

num_vocab = Vocab("0123456789", "")
en_vocab = Vocab([w for _, t in num_en for w in tok_en(t)], " ")
ko_vocab = Vocab([c for _, t in num_ko for c in tok_ko(t)], "")

print(len(num_vocab), len(en_vocab), len(ko_vocab))

ids = en_vocab.encode(tok_en("twelve thousand three hundred forty five"))
print(ids)
print(en_vocab.decode(ids))

ids = ko_vocab.encode(tok_ko("만 이천삼백사십오"))
print(ids)
print(ko_vocab.decode(ids))

ids = num_vocab.encode(tok_num("12345"))
print(ids)
print(num_vocab.decode(ids))

14 36 20
[1, 32, 30, 31, 15, 12, 11, 2]
twelve thousand three hundred forty five
[1, 6, 4, 15, 17, 9, 7, 8, 10, 13, 2]
만 이천삼백사십오
[1, 5, 6, 7, 8, 9, 2]
12345


In [5]:
import random
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

random.seed(42)

LANGS = {
    "num": (tok_num, num_vocab),
    "en": (tok_en, en_vocab),
    "ko": (tok_ko, ko_vocab),
}

num2en = dict(num_en)
num2ko = dict(num_ko)
shared = sorted(set(num2en) & set(num2ko))
en_ko = [(num2en[n], num2ko[n]) for n in shared]
print(len(en_ko))

def split(pairs, ratio=(0.8, 0.1, 0.1)):
    pairs = pairs[:]
    random.shuffle(pairs)
    a = int(len(pairs) * ratio[0])
    b = a + int(len(pairs) * ratio[1])
    return pairs[:a], pairs[a:b], pairs[b:]

base = {
    ("num", "en"): split(num_en),
    ("num", "ko"): split(num_ko),
    ("en", "ko"): split(en_ko),
}

data = {}
for (a, b), (tr, va, te) in base.items():
    data[(a, b)] = (tr, va, te)
    data[(b, a)] = tuple([(y, x) for x, y in part] for part in (tr, va, te))

for k, (tr, va, te) in data.items():
    print(k, len(tr), len(va), len(te), tr[0])

class PairDataset(Dataset):
    def __init__(self, pairs, src, tgt):
        s_tok, s_voc = LANGS[src]
        t_tok, t_voc = LANGS[tgt]
        self.data = [
            (torch.tensor(s_voc.encode(s_tok(x))), torch.tensor(t_voc.encode(t_tok(y))))
            for x, y in pairs
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        return self.data[i]

def collate(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=PAD)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD)
    return src, tgt

ds = PairDataset(data[("num", "en")][0], "num", "en")
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate)
src, tgt = next(iter(loader))
print(src.shape, tgt.shape)
print(src)
print(tgt)
for s, t in zip(src, tgt):
    print(num_vocab.decode(s.tolist()), "->", en_vocab.decode(t.tolist()))

12738
('num', 'en') 80000 10000 10000 ('962', 'nine hundred sixty two')
('en', 'num') 80000 10000 10000 ('nine hundred sixty two', '962')
('num', 'ko') 80000 10000 10000 ('12', '십이')
('ko', 'num') 80000 10000 10000 ('십이', '12')
('en', 'ko') 10190 1273 1275 ('eight thousand one hundred forty two', '팔천백사십이')
('ko', 'en') 10190 1273 1275 ('팔천백사십이', 'eight thousand one hundred forty two')
torch.Size([4, 12]) torch.Size([4, 17])
tensor([[ 1,  7, 11,  5,  7, 10,  6,  8,  6,  9, 13,  2],
        [ 1,  9,  7,  5, 10,  8, 13,  9,  4,  9,  2,  0],
        [ 1, 12,  7,  9,  8,  8,  4, 13,  5,  2,  0,  0],
        [ 1,  9,  4, 10,  7,  7,  4,  4,  6,  2,  0,  0]])
tensor([[ 1, 31,  4, 21, 15, 28, 16, 24, 15, 33, 13, 30, 34, 15, 10, 17,  2],
        [ 1, 11, 15, 29, 20, 16, 24, 15, 12, 17, 30, 11, 15, 11,  2,  0,  0],
        [ 1,  7, 31, 16, 11, 15, 12, 13, 30, 19, 20,  2,  0,  0,  0,  0,  0],
        [ 1, 10, 16, 24, 15, 29, 31, 30, 34,  2,  0,  0,  0,  0,  0,  0,  0]])
3713624259 -> three billio

In [6]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        lengths = (src != PAD).sum(1).cpu()
        emb = self.embedding(src)
        packed = pack_padded_sequence(emb, lengths, batch_first=True, enforce_sorted=False)
        out, hidden = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        return out, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, token, hidden):
        emb = self.embedding(token.unsqueeze(1))
        out, hidden = self.rnn(emb, hidden)
        logits = self.fc(out.squeeze(1))
        return logits, hidden

class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, emb_dim=128, hid_dim=256, n_layers=2, dropout=0.2):
        super().__init__()
        self.encoder = Encoder(src_vocab, emb_dim, hid_dim, n_layers, dropout)
        self.decoder = Decoder(tgt_vocab, emb_dim, hid_dim, n_layers, dropout)
        self.tgt_vocab = tgt_vocab

    def forward(self, src, tgt, teacher_forcing=0.5):
        B, T = tgt.shape
        _, hidden = self.encoder(src)
        outputs = torch.zeros(B, T - 1, self.tgt_vocab, device=src.device)
        token = tgt[:, 0]
        for t in range(1, T):
            logits, hidden = self.decoder(token, hidden)
            outputs[:, t - 1] = logits
            if random.random() < teacher_forcing:
                token = tgt[:, t]
            else:
                token = logits.argmax(1)
        return outputs

model = Seq2Seq(len(num_vocab), len(en_vocab)).to(device)
print(model)
print(sum(p.numel() for p in model.parameters()))

src, tgt = src.to(device), tgt.to(device)
out = model(src, tgt)
print(src.shape, tgt.shape, out.shape)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(14, 128, padding_idx=0)
    (rnn): GRU(128, 256, num_layers=2, batch_first=True, dropout=0.2)
  )
  (decoder): Decoder(
    (embedding): Embedding(36, 128, padding_idx=0)
    (rnn): GRU(128, 256, num_layers=2, batch_first=True, dropout=0.2)
    (fc): Linear(in_features=256, out_features=36, bias=True)
  )
)
1398052
torch.Size([4, 12]) torch.Size([4, 17]) torch.Size([4, 16, 36])


In [7]:
import time

def make_loader(src, tgt, part, batch_size=128, shuffle=False):
    idx = {"train": 0, "val": 1, "test": 2}[part]
    ds = PairDataset(data[(src, tgt)][idx], src, tgt)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)

def run_epoch(model, loader, optimizer=None, teacher_forcing=0.5):
    train = optimizer is not None
    model.train() if train else model.eval()
    total, count = 0, 0
    with torch.set_grad_enabled(train):
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            out = model(src, tgt, teacher_forcing if train else 0)
            loss = criterion(out.reshape(-1, out.size(-1)), tgt[:, 1:].reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total += loss.item() * src.size(0)
            count += src.size(0)
    return total / count

def train_model(src, tgt, epochs=10, lr=1e-3):
    s_voc, t_voc = LANGS[src][1], LANGS[tgt][1]
    model = Seq2Seq(len(s_voc), len(t_voc)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = make_loader(src, tgt, "train", shuffle=True)
    val_loader = make_loader(src, tgt, "val")
    for epoch in range(1, epochs + 1):
        start = time.time()
        tr = run_epoch(model, train_loader, optimizer)
        va = run_epoch(model, val_loader)
        print(f"[{src}->{tgt}] epoch {epoch:2d}  train {tr:.4f}  val {va:.4f}  {time.time() - start:.1f}s")
    return model

model = train_model("num", "en", epochs=10)

[num->en] epoch  1  train 0.8514  val 0.2401  13.6s
[num->en] epoch  2  train 0.0819  val 0.0500  15.8s
[num->en] epoch  3  train 0.0281  val 0.0207  13.5s
[num->en] epoch  4  train 0.0183  val 0.0202  12.9s
[num->en] epoch  5  train 0.0094  val 0.0159  13.4s
[num->en] epoch  6  train 0.0088  val 0.0090  17.0s
[num->en] epoch  7  train 0.0073  val 0.0092  15.3s
[num->en] epoch  8  train 0.0062  val 0.0150  16.9s
[num->en] epoch  9  train 0.0071  val 0.0064  15.5s
[num->en] epoch 10  train 0.0039  val 0.0055  17.4s


In [8]:
@torch.no_grad()
def greedy_decode(model, src, max_len=30):
    model.eval()
    _, hidden = model.encoder(src)
    B = src.size(0)
    token = torch.full((B,), SOS, dtype=torch.long, device=src.device)
    result = []
    finished = torch.zeros(B, dtype=torch.bool, device=src.device)
    for _ in range(max_len):
        logits, hidden = model.decoder(token, hidden)
        token = logits.argmax(1)
        result.append(token)
        finished |= token == EOS
        if finished.all():
            break
    return torch.stack(result, 1)

def translate(model, text, src, tgt):
    s_tok, s_voc = LANGS[src]
    t_voc = LANGS[tgt][1]
    x = torch.tensor([s_voc.encode(s_tok(text))], device=device)
    return t_voc.decode(greedy_decode(model, x)[0].tolist())

def evaluate(model, src, tgt, part="test", show_errors=5):
    t_voc = LANGS[tgt][1]
    s_voc = LANGS[src][1]
    loader = make_loader(src, tgt, part, batch_size=512)
    correct, total, errors = 0, 0, []
    for x, y in loader:
        x = x.to(device)
        pred = greedy_decode(model, x)
        for xi, yi, pi in zip(x.tolist(), y.tolist(), pred.tolist()):
            p, g = t_voc.decode(pi), t_voc.decode(yi)
            if p == g:
                correct += 1
            elif len(errors) < show_errors:
                errors.append((s_voc.decode(xi), g, p))
            total += 1
    acc = correct / total
    print(f"[{src}->{tgt}] {part} accuracy: {acc:.4f} ({correct}/{total})")
    for s, g, p in errors:
        print(f"  {s}\n    answer: {g}\n    pred:   {p}")
    return acc

evaluate(model, "num", "en")
for n in ["0", "7", "15", "100", "12345", "1000001", "9876543210"]:
    print(n, "->", translate(model, n, "num", "en"))

[num->en] test accuracy: 0.9964 (9964/10000)
  114048881
    answer: one hundred fourteen million forty eight thousand eight hundred eighty one
    pred:   one hundred fourteen million forty four thousand eight hundred eighty one
  7427225845
    answer: seven billion four hundred twenty seven million two hundred twenty five thousand eight hundred forty five
    pred:   seven billion four hundred twenty seven million two hundred fifty five thousand eight hundred forty five
  8200047531
    answer: eight billion two hundred million forty seven thousand five hundred thirty one
    pred:   eight billion two hundred million four thousand five hundred thirty one
  5483400716
    answer: five billion four hundred eighty three million four hundred thousand seven hundred sixteen
    pred:   five billion four hundred eighty three million four hundred thousand one hundred sixteen
  991474054
    answer: nine hundred ninety one million four hundred seventy four thousand fifty four
    pred:   nin

In [9]:
CONFIG = dict(emb_dim=128, hid_dim=256, n_layers=2, dropout=0.2)
EPOCHS = {("num", "en"): 10, ("en", "num"): 10, ("num", "ko"): 10,
          ("ko", "num"): 10, ("en", "ko"): 30, ("ko", "en"): 30}

models, results = {}, {}
for (src, tgt), ep in EPOCHS.items():
    m = train_model(src, tgt, epochs=ep)
    results[f"{src}2{tgt}"] = evaluate(m, src, tgt, show_errors=3)
    models[f"{src}2{tgt}"] = m

print()
for k, v in results.items():
    print(f"{k:8s} {v:.4f}")

torch.save({
    "config": CONFIG,
    "vocabs": {k: (v.itos, v.joiner) for k, v in [("num", num_vocab), ("en", en_vocab), ("ko", ko_vocab)]},
    "models": {k: m.state_dict() for k, m in models.items()},
}, "num2text_models.pt")

[num->en] epoch  1  train 0.8321  val 0.2854  16.8s
[num->en] epoch  2  train 0.0792  val 0.0556  18.0s
[num->en] epoch  3  train 0.0282  val 0.1298  14.8s
[num->en] epoch  4  train 0.0274  val 0.0685  13.6s
[num->en] epoch  5  train 0.0093  val 0.0071  14.2s
[num->en] epoch  6  train 0.0111  val 0.0271  13.6s
[num->en] epoch  7  train 0.0054  val 0.0115  13.1s
[num->en] epoch  8  train 0.0111  val 0.0151  13.4s
[num->en] epoch  9  train 0.0061  val 0.0041  13.2s
[num->en] epoch 10  train 0.0033  val 0.0048  13.2s
[num->en] test accuracy: 0.9968 (9968/10000)
  4023710420
    answer: four billion twenty three million seven hundred ten thousand four hundred twenty
    pred:   four billion twenty three million seven hundred one thousand four hundred twenty
  1821715626
    answer: one billion eight hundred twenty one million seven hundred fifteen thousand six hundred twenty six
    pred:   one billion eight hundred twenty seven million one hundred seventy five thousand six hundred twenty 

In [10]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

PAD, SOS, EOS, UNK = 0, 1, 2, 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Vocab:
    def __init__(self, itos, joiner):
        self.itos = itos
        self.stoi = {t: i for i, t in enumerate(itos)}
        self.joiner = joiner

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens):
        return [SOS] + [self.stoi.get(t, UNK) for t in tokens] + [EOS]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == EOS:
                break
            if i in (PAD, SOS):
                continue
            out.append(self.itos[i])
        return self.joiner.join(out)

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)

    def forward(self, src):
        lengths = (src != PAD).sum(1).cpu()
        packed = pack_padded_sequence(self.embedding(src), lengths, batch_first=True, enforce_sorted=False)
        _, hidden = self.rnn(packed)
        return None, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers, batch_first=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, token, hidden):
        out, hidden = self.rnn(self.embedding(token.unsqueeze(1)), hidden)
        return self.fc(out.squeeze(1)), hidden

class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.encoder = Encoder(src_vocab, emb_dim, hid_dim, n_layers, dropout)
        self.decoder = Decoder(tgt_vocab, emb_dim, hid_dim, n_layers, dropout)

TOKENIZERS = {"num": list, "en": str.split, "ko": list}

ckpt = torch.load("num2text_models.pt", map_location=device)
VOCABS = {k: Vocab(itos, joiner) for k, (itos, joiner) in ckpt["vocabs"].items()}
MODELS = {}
for name, state in ckpt["models"].items():
    src, tgt = name.split("2")
    m = Seq2Seq(len(VOCABS[src]), len(VOCABS[tgt]), **ckpt["config"]).to(device)
    m.load_state_dict(state)
    m.eval()
    MODELS[name] = m

@torch.no_grad()
def translate(text, src, tgt, max_len=30):
    model = MODELS[f"{src}2{tgt}"]
    s_voc, t_voc = VOCABS[src], VOCABS[tgt]
    x = torch.tensor([s_voc.encode(TOKENIZERS[src](text))], device=device)
    _, hidden = model.encoder(x)
    token = torch.tensor([SOS], device=device)
    out = []
    for _ in range(max_len):
        logits, hidden = model.decoder(token, hidden)
        token = logits.argmax(1)
        if token.item() == EOS:
            break
        out.append(token.item())
    return t_voc.decode(out)

def num2en(s): return translate(s, "num", "en")
def en2num(s): return translate(s, "en", "num")
def num2ko(s): return translate(s, "num", "ko")
def ko2num(s): return translate(s, "ko", "num")
def en2ko(s): return translate(s, "en", "ko")
def ko2en(s): return translate(s, "ko", "en")

print(num2en("12345"))
print(en2num("twelve thousand three hundred forty five"))
print(num2ko("12345"))
print(ko2num("만 이천삼백사십오"))
print(en2ko("twelve thousand three hundred forty five"))
print(ko2en("만 이천삼백사십오"))

for n in ["0", "7", "19", "100", "1000001", "2026112024", "9876543210"]:
    en, ko = num2en(n), num2ko(n)
    print(f"{n:>11} | {en} | {ko} | {en2num(en)} | {ko2num(ko)}")

twelve thousand three hundred forty five
12345
만 이천삼백사십오
12345
만 이천삼백사십오
twelve thousand three hundred forty five
          0 | zero | 영 | 0 | 0
          7 | seven | 칠 | 7 | 7
         19 | nineteen | 십구 | 19 | 19
        100 | one hundred | 백 | 100 | 100
    1000001 | one million one hundred one | 백만 일 | 110101 | 1000001
 2026112024 | two billion twenty six million one hundred twelve thousand twenty four | 이십억 이천육백십일만 이천이십사 | 2026112024 | 2026112024
 9876543210 | nine billion eight hundred seventy six million five hundred forty three thousand two hundred ten | 구십팔억 칠천육백오십사만 삼천이백십 | 9876543210 | 9876543210
